# Cross-Domain Trace Analysis (S1 / S2 / S3)

Exploratory extension, **not** part of the core Arm A/B research question -- see
`docs/equivalence/cross-domain-analysis-design.md` for the full proposed scope
(topological analysis, shared-characteristic analysis, category-theoretic
scaffolding) and `docs/DECISIONS.md` #11/#12/#14 for what was actually built and
why.

Covers **all five S1 topology methods** now (S1.1 disconnectivity, S1.2
persistent homology, S1.3 trap rate, S1.4 Mapper, S1.5 Forman-Ricci curvature),
plus S2 (shared characteristics) and the lite S3.1/3.2 (status transition
matrix). `ripser`/`persim` (S1.2), `networkx`/`GraphRicciCurvature` (S1.5) are
now installed; S1.4's Mapper is implemented directly (`analysis/mapper.py`)
rather than via `kmapper`, since the doc calls for graph-adjacency clustering
within each filter interval, not `kmapper`'s default Euclidean clusterer.
`giotto-tda` is still not used -- `ripser`+`persim` cover the doc's scoped-down
"(a) feature embedding first" request without it.

**S1.1, S1.2, and S1.4 are population comparisons** (`POP_SAMPLE_SIZE`
instances per domain, fixed seed) rather than single hand-picked instances --
an earlier single-/dual-instance version of each existed first, and was
replaced after review: a picture of one or two instances doesn't establish
that a difference holds generally, and says little about the *nature* of
either domain's search space as a population property. **S1.5 keeps its
single-instance plot** (its value is showing *where* one tree is locally
bottlenecked, which pooling would erase) and adds a separate population
view alongside it. Population comparisons use scalar summary statistics
(bar counts, total persistence, fragmentation ratio, mean curvature) plus a
Mann-Whitney U test, not full persistence-landscape/image averaging --
`persim` has `PersLandscapeApprox`/`PersistenceImager` for that if more
resolution is wanted later; the scalar-summary route was chosen to avoid
taking on an unfamiliar library API under time pressure, not because
landscapes wouldn't be better.

Everything here reads `results/traces/*.csv` through the `analysis/` package --
the S2/S1.3/S3.1/3.2 cells reuse the same functions `scripts/analyze_traces.py`
uses; S1.1/1.2/1.4/1.5 are new (this session) and only live here so far, not
in the CLI script.

**Treat this as directional, not confirmatory** -- the sample is whatever's in
`results/traces/` right now (a mix of real downloaded proteins and synthetic
generated sequences for HP, a subset of the map suite for Sokoban), not a locked
benchmark suite collected under one fixed protocol, and `POP_SAMPLE_SIZE=30` is
itself a modest population, not the full corpus.

In [ ]:
import sys
import statistics
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from analysis.trace_io import domain_of, read_trace
from analysis.shared_characteristics import branching_factors, feasibility_ratios, plateau_run_lengths
from analysis.topology_lite import trap_rate
from analysis.category_lite import eigen_spectrum, kl_divergence, transition_counts, transition_matrix

TRACE_DIR = REPO_ROOT / "results" / "traces"
files = sorted(TRACE_DIR.glob("*_trace.csv"))
by_domain = {"sokoban": [], "hp_lattice": []}
for f in files:
    by_domain[domain_of(f)].append(f)
print(f"{len(files)} trace files: {len(by_domain['sokoban'])} sokoban, {len(by_domain['hp_lattice'])} hp_lattice")

## One streaming pass per domain

The trace corpus is large (millions of rows across both domains), so this reads
each file once and keeps only the *derived* aggregates (branching factors,
feasibility ratios, plateau lengths, trap counts, transition counts) -- not the
raw rows -- mirroring `scripts/analyze_traces.py`'s own memory-light approach.
Takes roughly a minute for the full corpus.

In [ ]:
def aggregate_domain(paths):
    agg = {
        "branching": [], "feasibility": [], "plateau": [],
        "trap_expanded": 0, "trap_hits": 0,
        "transition_counts": {},
    }
    for p in paths:
        rows = list(read_trace(p))
        agg["branching"].extend(branching_factors(rows))
        agg["feasibility"].extend(feasibility_ratios(rows))
        agg["plateau"].extend(plateau_run_lengths(rows))
        t = trap_rate(rows)
        agg["trap_expanded"] += t["n_expanded"]
        agg["trap_hits"] += t["n_trap"]
        for k, v in transition_counts(rows).items():
            agg["transition_counts"][k] = agg["transition_counts"].get(k, 0) + v
    return agg

agg_by_domain = {d: aggregate_domain(paths) for d, paths in by_domain.items()}
for d, agg in agg_by_domain.items():
    print(f"{d}: {len(agg['branching'])} nodes-with-successors, "
          f"trap {agg['trap_hits']}/{agg['trap_expanded']}")

## S2 -- Shared characteristics without reduction

These three read columns the trace already has (`n_legal_successors`, `n_pruned`,
`f`, `status`, `timestamp_order`) -- no extra instrumentation, no domain
translation needed, since both solvers log the same column names for
structurally analogous concepts (a "successor" in Sokoban is a legal push; in HP
it's a legal next-monomer placement).

- **S2.1 branching factor** -- `n_legal_successors` per expanded node.
- **S2.2 feasibility ratio** -- `n_legal_successors / (n_legal_successors +
  n_pruned)`, i.e. what fraction of a node's structurally-generated candidates
  survive.
- **S2.4 plateau run length** -- consecutive expansions (by `timestamp_order`)
  where `f` doesn't improve -- a coarse "is the search making progress" signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (domain, agg) in zip(axes, agg_by_domain.items()):
    bf = agg["branching"]
    color = "steelblue" if domain == "sokoban" else "darkorange"
    ax.hist(bf, bins=range(0, max(bf) + 2), color=color)
    ax.set_title(f"{domain}\nmean={statistics.mean(bf):.2f}  median={statistics.median(bf)}")
    ax.set_xlabel("n_legal_successors")
fig.suptitle("S2.1 -- branching factor spectra")
plt.tight_layout()
plt.show()

Sokoban's branching factor is structurally larger (up to 4 push directions,
compounded across however many crates are simultaneously pushable) than HP's
(at most 4 lattice neighbors minus the fixed backbone predecessor, so
effectively <=3) -- expect Sokoban's mean to sit meaningfully higher. That's a
structural fact about the two move-generation rules, not a claim about search
quality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (domain, agg) in zip(axes, agg_by_domain.items()):
    fr = agg["feasibility"]
    color = "steelblue" if domain == "sokoban" else "darkorange"
    ax.hist(fr, bins=20, color=color)
    ax.set_title(f"{domain}\nmean={statistics.mean(fr):.3f}  median={statistics.median(fr):.3f}")
    ax.set_xlabel("feasibility ratio")
fig.suptitle("S2.2 -- feasibility ratio")
plt.tight_layout()
plt.show()

**Read this one carefully -- it's not measuring the same *kind* of rejection in
both domains.** Sokoban's "pruned" is a domain-constraint violation (a provable
dead square, `board.is_dead()`) discovered *after* a structurally legal push is
generated. HP's is a search-optimality rejection (the bound can't beat the
current incumbent) -- HP's actual domain constraint (self-avoidance) is enforced
silently at candidate generation and never shows up as a rejection at all. So
HP's near-1.0 feasibility ratio doesn't mean "HP wastes fewer candidates than
Sokoban" -- it means most of HP's real filtering happens somewhere this ratio
can't see. Full detail: `bnb.py`'s module docstring.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (domain, agg) in zip(axes, agg_by_domain.items()):
    pl = agg["plateau"]
    color = "steelblue" if domain == "sokoban" else "darkorange"
    ax.hist(pl, bins=30, color=color)
    ax.set_yscale("log")
    ax.set_title(f"{domain}\nmean={statistics.mean(pl):.1f}  median={statistics.median(pl)}  max={max(pl)}")
    ax.set_xlabel("plateau run length (consecutive non-improving expansions)")
fig.suptitle("S2.4 -- plateau/shoulder run lengths (log-scale y)")
plt.tight_layout()
plt.show()

## S1.3 -- Trap rate (index-0 critical cells)

A "trap" here is `all_pruned`: an expanded node where every structurally-legal
successor got rejected -- a genuine dead end, the direct analog of a
Morse-theory index-0 critical cell (a local optimum the search can only leave
via a worse state). This is the one topology-flavored signal that's essentially
free: it's already computed live during search, this just aggregates it.

In [ ]:
for domain, agg in agg_by_domain.items():
    rate = agg["trap_hits"] / agg["trap_expanded"] if agg["trap_expanded"] else float("nan")
    print(f"{domain}: {agg['trap_hits']}/{agg['trap_expanded']} expanded nodes were traps ({rate:.4f})")

## S1.1, S1.2, S1.4 -- population-level, not single-instance

Earlier versions of this section plotted one or two hand-picked instances
per domain for S1.1, S1.2, and S1.4. That's a case study, not a population
claim -- "Sokoban's Mapper graph is more fragmented than HP's" on *one pair*
of instances chosen partly for plot legibility doesn't establish that this
holds generally, and doesn't say much about the *nature* of either domain's
search space as a whole. This section replaces those three with population
comparisons: `POP_SAMPLE_SIZE` instances per domain (fixed seed, sampled
without replacement from everything in `results/traces/`), one pass per
instance computing all three methods' outputs, aggregated into
distributions/curves and compared with a Mann-Whitney U test where a formal
comparison makes sense. (S1.5 stays single-instance below, per request --
its point is showing *where* a search tree is locally bottlenecked, which a
population summary would flatten away; a population *view* of it is added
afterward instead of replacing the instance one.)

In [ ]:
import random

POP_SAMPLE_SIZE = 30
POP_SEED = 0

def population_sample(paths, n=POP_SAMPLE_SIZE, seed=POP_SEED):
    if len(paths) <= n:
        return paths
    return random.Random(seed).sample(paths, n)

pop_by_domain = {d: population_sample(paths) for d, paths in by_domain.items()}
for d, paths in pop_by_domain.items():
    print(f"{d}: {len(paths)} instances sampled for population-level S1.1/S1.2/S1.4 analysis")

In [ ]:
from analysis.topology_lite import disconnectivity_curve_normalized
from analysis.persistence import point_cloud, persistence_diagrams, diagram_summary
from analysis.mapper import fragmentation_ratio

pop_data = {
    d: {"disc_curves": [], "h0_bars": [], "h1_bars": [], "h0_total_pers": [], "h1_total_pers": [], "frag_ratios": []}
    for d in pop_by_domain
}

for domain, paths in pop_by_domain.items():
    print(f"computing population S1 metrics for {domain} ({len(paths)} instances)...")
    for path in paths:
        rows = list(read_trace(path))

        curve = disconnectivity_curve_normalized(rows)
        if curve:
            pop_data[domain]["disc_curves"].append(curve)

        pts = point_cloud(rows, max_points=500)
        summary = diagram_summary(persistence_diagrams(pts))
        pop_data[domain]["h0_bars"].append(summary[0]["n_bars"])
        pop_data[domain]["h1_bars"].append(summary[1]["n_bars"])
        pop_data[domain]["h0_total_pers"].append(summary[0]["total_persistence"])
        pop_data[domain]["h1_total_pers"].append(summary[1]["total_persistence"])

        frag = fragmentation_ratio(rows)
        if frag is not None:
            pop_data[domain]["frag_ratios"].append(frag)
    n_excluded_curves = len(paths) - len(pop_data[domain]["disc_curves"])
    print(f"  done: {len(pop_data[domain]['h0_bars'])} instances processed "
          f"({n_excluded_curves} excluded from the S1.1 fan chart -- degenerate/constant-f trace, "
          f"not a bug, see analysis/topology_lite.py::disconnectivity_curve_normalized)")

### S1.1 population -- disconnectivity fan chart

Each instance's curve is normalized to [0, 1] on both axes (tau by that
instance's own f-range, component count by its own starting count) before
plotting, so instances of very different sizes/scales become directly
comparable -- same question for every curve: "as tau grows from the
minimum to the maximum f seen, how fast does the search-expansion subgraph
collapse toward one connected piece?" Faint lines = individual instances,
bold = the per-domain mean.

In [ ]:
def plot_fan(curves, ax, color, label):
    if not curves:
        return
    grid = [t for t, _ in curves[0]]
    ys = np.array([[y for _, y in c] for c in curves])
    for row in ys:
        ax.plot(grid, row, color=color, alpha=0.15, linewidth=1)
    ax.plot(grid, ys.mean(axis=0), color=color, linewidth=2.5, label=f"{label} mean (n={len(curves)})")

fig, ax = plt.subplots(figsize=(7, 5))
plot_fan(pop_data["sokoban"]["disc_curves"], ax, "steelblue", "sokoban")
plot_fan(pop_data["hp_lattice"]["disc_curves"], ax, "darkorange", "hp_lattice")
ax.set_xlabel("normalized tau (0 = min f, 1 = max f)")
ax.set_ylabel("normalized component count (1.0 = count at tau=min)")
ax.set_title("S1.1 population -- disconnectivity fan chart")
ax.legend()
plt.show()

### S1.2 population -- persistence summary statistics

Rather than averaging full persistence diagrams (landscapes/persistence
images support that properly, `persim` has `PersLandscapeApprox`/
`PersistenceImager` for it) this uses simpler scalar summaries per
instance -- bar count and total persistence (sum of lifetimes) for H0 and
H1 -- which is enough to run an actual two-sample test across the
population without taking on an unfamiliar library API under time
pressure (this project has already been burned twice doing that). H0's one
infinite-death bar is capped at that diagram's own max finite value before
summing (`analysis/persistence.py::diagram_summary`).

In [ ]:
from scipy import stats

def compare_metric(key, label, ax):
    data = {d: pop_data[d][key] for d in pop_data}
    ax.boxplot(list(data.values()), tick_labels=list(data.keys()))
    a, b = data["sokoban"], data["hp_lattice"]
    try:
        _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        ax.set_title(f"{label}\nMann-Whitney U p={p:.4g}")
    except ValueError:
        ax.set_title(f"{label}\n(test undefined -- no variation in one group)")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
compare_metric("h0_bars", "H0 bar count", axes[0][0])
compare_metric("h1_bars", "H1 bar count", axes[0][1])
compare_metric("h0_total_pers", "H0 total persistence", axes[1][0])
compare_metric("h1_total_pers", "H1 total persistence", axes[1][1])
fig.suptitle(f"S1.2 population -- persistence summary statistics "
             f"(n={len(pop_data['sokoban']['h0_bars'])} sokoban, {len(pop_data['hp_lattice']['h0_bars'])} hp_lattice)")
plt.tight_layout()
plt.show()

### S1.4 population -- fragmentation ratio

`fragmentation_ratio` = Mapper clusters / visited nodes, `n_intervals=15`
fixed across every instance and both domains (the single-instance version's
per-domain-tuned interval counts made instances incomparable by
construction). This is the population-level version of the ~230x
node-count gap noted before: does one domain's search graph genuinely
fragment more densely within a narrow f-band than the other's, across many
instances, not just the two picked for a legible picture?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
compare_metric("frag_ratios", "fragmentation ratio (clusters / visited nodes)", ax)
plt.show()
for d in pop_data:
    vals = pop_data[d]["frag_ratios"]
    print(f"{d}: mean={np.mean(vals):.4f} median={np.median(vals):.4f} n={len(vals)}")

## S1.5 -- Graph curvature (Forman-Ricci)

Forman-Ricci curvature (no optimal-transport solver needed -- cheaper than
Ollivier-Ricci, per the doc's own prototype-scope note) via
`GraphRicciCurvature`, directly on the reconstructed graph (`parent_id`
edges) -- no embedding, no `f` needed, curvature comes purely from local
neighborhood overlap. Negative-curvature edges are bottlenecks (the only
route between two regions); positive-curvature edges sit in well-connected,
redundant regions.

Sampled to the first ~300 visited nodes by expansion order (full ancestor
chains included, so the sample is guaranteed connected) -- full traces are
far too large to lay out legibly as a graph.

In [ ]:
from analysis.curvature import sample_connected_subgraph, build_graph, forman_curvature

fig, axes = plt.subplots(2, 2, figsize=(12, 9), gridspec_kw={"height_ratios": [2, 1]})
for col, (domain, paths) in enumerate(by_domain.items()):
    largest = max(paths, key=lambda p: p.stat().st_size)
    rows = list(read_trace(largest))
    sub = sample_connected_subgraph(rows, max_nodes=300)
    g = build_graph(sub)
    curv = forman_curvature(g)

    ax_graph, ax_hist = axes[0][col], axes[1][col]
    pos = nx.spring_layout(g, seed=0)
    edge_vals = [curv.get(e, curv.get((e[1], e[0]))) for e in g.edges]
    nx.draw_networkx_nodes(g, pos, node_size=15, node_color="black", ax=ax_graph)
    edges_drawn = nx.draw_networkx_edges(
        g, pos, edge_color=edge_vals, edge_cmap=plt.cm.coolwarm, width=1.5, ax=ax_graph,
    )
    ax_graph.set_title(f"{domain}: {largest.name}\n{g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
    ax_graph.axis("off")
    plt.colorbar(edges_drawn, ax=ax_graph, label="Forman curvature", shrink=0.7)

    vals = list(curv.values())
    ax_hist.hist(vals, bins=20, color="steelblue" if domain == "sokoban" else "darkorange")
    ax_hist.set_title(f"mean={statistics.mean(vals):.2f}  min={min(vals):.1f}  max={max(vals):.1f}")
    ax_hist.set_xlabel("edge Forman curvature")
fig.suptitle("S1.5 -- Forman-Ricci curvature (first ~300 nodes by expansion order)")
plt.tight_layout()
plt.show()

### S1.5 population -- mean curvature across instances

The instance plot above shows *where* one search tree is bottlenecked
(negative curvature) vs. well-connected (positive) -- useful, but it's one
tree, and pooling instances into it would erase exactly what makes it
useful. This is the complementary population view: one mean-edge-curvature
scalar per instance (same `sample_connected_subgraph`/~300-node window,
reused via `analysis.curvature.mean_curvature`), pooled across the same
`POP_SAMPLE_SIZE`-instance population as S1.1/S1.2/S1.4, to check whether
the instance plot's bottleneck-heavy look is a systematic domain property
or that one instance's idiosyncrasy.

In [ ]:
from analysis.curvature import mean_curvature

pop_curvature = {d: [] for d in pop_by_domain}
for domain, paths in pop_by_domain.items():
    print(f"computing population S1.5 curvature for {domain} ({len(paths)} instances)...")
    for path in paths:
        rows = list(read_trace(path))
        mc = mean_curvature(rows)
        if mc is not None:
            pop_curvature[domain].append(mc)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.boxplot(list(pop_curvature.values()), tick_labels=list(pop_curvature.keys()))
a, b = pop_curvature["sokoban"], pop_curvature["hp_lattice"]
try:
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    ax.set_title(f"S1.5 population -- mean edge curvature per instance\nMann-Whitney U p={p:.4g}")
except ValueError:
    ax.set_title("S1.5 population -- mean edge curvature per instance")
ax.set_ylabel("mean Forman curvature (per-instance average)")
plt.show()
for d in pop_curvature:
    vals = pop_curvature[d]
    print(f"{d}: mean={np.mean(vals):.4f} median={np.median(vals):.4f} n={len(vals)}")

## S3.1 / S3.2 -- Status transition matrix

`docs/equivalence/cross-domain-analysis-design.md` proposes a 5-object category
(`FRONTIER, EXPANDED, PRUNED, TRAP, GOAL`) with live status logging at every
state touch. What's actually built here is a lighter, **post-hoc** version:
`status` is read straight from the trace (already logged live during search),
and transitions are derived by joining each row to its parent's status via
`parent_id` -- no new instrumentation, but two real limitations follow from
that shortcut:

- **Only 3 of the 5 objects are ever observed**: EXPANDED, PRUNED, GOAL. There's
  no live FRONTIER tracking (states generated but neither expanded nor pruned),
  and TRAP isn't a `status` value (see S1.3 above -- it's the separate
  `all_pruned` column instead).
- **PRUNED and GOAL are terminal**: nothing is ever logged as the parent of a
  node whose own parent was already pruned or a goal, so those two rows are
  *structurally* all-zeros in both domains' matrices. That matters below.

In [ ]:
matrices = {}
for domain, agg in agg_by_domain.items():
    labels, mat = transition_matrix(agg["transition_counts"])
    matrices[domain] = (labels, mat)
    print(f"{domain} (labels={labels}):")
    for label, row in zip(labels, mat):
        print(f"  {label:>10}: " + " ".join(f"{v:.3f}" for v in row))
    print()

### KL divergence and eigenspectrum

KL divergence compares each matrix row (a probability distribution over "what
does this status transition to next") between domains. **The `goal` and
`pruned` rows are always vacuous** -- both domains' matrices have all-zero rows
there (terminal states never have outgoing transitions, so there's no
distribution to compare on either side), and KL between "no data" and "no data"
trivially returns 0. That is *not* a finding that the two domains behave
identically for those states -- it's a structural non-comparison, flagged
explicitly below rather than printed as a bare, misleading `0.0000`
(this exact confusion came up live while building this notebook -- see
`docs/DECISIONS.md` for the fix).

The one row with real content -- `expanded` -- is where the actual comparison
lives.

In [ ]:
(label_a, (labels_a, mat_a)), (label_b, (labels_b, mat_b)) = matrices.items()
assert labels_a == labels_b, f"status sets differ: {labels_a} vs {labels_b} -- see docs/DECISIONS.md #12"

for i, label in enumerate(labels_a):
    if sum(mat_a[i]) == 0 and sum(mat_b[i]) == 0:
        print(f"KL({label_a}[{label}] || {label_b}[{label}]) = N/A "
              f"(vacuous -- no outgoing transitions from '{label}' in either domain)")
        continue
    kl = kl_divergence(mat_a[i], mat_b[i])
    print(f"KL({label_a}[{label}] || {label_b}[{label}]) = {kl:.4f}")

print()
print(f"{label_a} eigenspectrum:  ", [f"{v.real:.3f}" for v in eigen_spectrum(mat_a)])
print(f"{label_b} eigenspectrum:  ", [f"{v.real:.3f}" for v in eigen_spectrum(mat_b)])

### Interpreting the divergence

The `expanded` row's KL divergence is the real signal: it captures how
differently each domain's search "spends" its expanded nodes across
{keeps expanding, reaches a goal, gets pruned}. Read the printed row
probabilities above directly -- if HP's `expanded -> goal` probability is much
higher than Sokoban's, that's very likely a **tree-shape artifact rather than a
search-quality difference**: every HP branch has exactly the same depth (chain
length - 1 placements to a complete fold), so many nodes near the end of the
tree are mechanically one step from a leaf, whereas Sokoban's tree has variable
depth and several ways to fail before reaching a goal. Don't read this as "HP
prunes worse" without controlling for that.

## Connectivity-pruning proof of concept: before/after

`docs/DECISIONS.md` #15 built `connectivity_prune` as an opt-in
domain-constraint deadlock check (`bnb.py`), the HP analog of Sokoban's
`is_dead()`. Node-count/wall-clock benchmarking found it marginal (~0.1-0.2%
fewer `nodes_expanded`, roughly a wash on time). This closes the loop from
the other direction: does that marginal effect show up in S1.3's trap rate
or S3.1's transition matrix, using the *same* 18 instances traced both ways
-- `results/traces/` (bound-only, already aggregated above) vs
`results/traces_connectivity/` (bound+connectivity, generated separately so
it doesn't get folded into the main corpus and blur the comparison)?

In [ ]:
CONN_DIR = REPO_ROOT / "results" / "traces_connectivity"
conn_files = sorted(CONN_DIR.glob("*_trace.csv"))
pairs = [(TRACE_DIR / f.name, f) for f in conn_files if (TRACE_DIR / f.name).exists()]
print(f"{len(pairs)} matched instance pairs (same sequence, traced with and without connectivity_prune)")

def aggregate_pair(paths):
    agg = {"trap_expanded": 0, "trap_hits": 0, "transition_counts": {}}
    for p in paths:
        rows = list(read_trace(p))
        t = trap_rate(rows)
        agg["trap_expanded"] += t["n_expanded"]
        agg["trap_hits"] += t["n_trap"]
        for k, v in transition_counts(rows).items():
            agg["transition_counts"][k] = agg["transition_counts"].get(k, 0) + v
    return agg

agg_baseline = aggregate_pair([b for b, c in pairs])
agg_conn = aggregate_pair([c for b, c in pairs])

print("--- S1.3 trap rate ---")
for label, agg in [("bound-only (baseline)", agg_baseline), ("bound+connectivity", agg_conn)]:
    rate = agg["trap_hits"] / agg["trap_expanded"] if agg["trap_expanded"] else float("nan")
    print(f"  {label}: {agg['trap_hits']}/{agg['trap_expanded']} ({rate:.4f})")

print()
print("--- S3.1 transition matrix ---")
for label, agg in [("bound-only (baseline)", agg_baseline), ("bound+connectivity", agg_conn)]:
    labels, mat = transition_matrix(agg["transition_counts"])
    print(f"  {label} (labels={labels}):")
    for lbl, row in zip(labels, mat):
        print(f"    {lbl:>10}: " + " ".join(f"{v:.4f}" for v in row))


**The trap rate roughly halves (see the numbers printed above), while the
`expanded -> pruned` transition probability barely moves.** Those aren't in
tension -- they're measuring different things. Connectivity pruning catches
some dead ends *earlier*, at an ancestor node, via a cheap local check,
before the search would otherwise have expanded several more descendants
and only then discovered the same dead end the slow way via the bound
(logged as a trap on some *descendant*, not the ancestor). That lines up
with the ~0.1-0.2% `nodes_expanded` reduction: fewer wasted expansions
downstream of a now-earlier-caught deadlock, even though the *total* count
of rejected candidates (pruned-fraction) barely changes -- it's the same
amount of rejection, arriving sooner.

**It does not close the gap toward Sokoban's ~59% pruned-fraction, and
wasn't expected to** (`docs/DECISIONS.md` #15) -- HP's bound already
implicitly captures most of what a domain-constraint check would add on
top, unlike Sokoban's heuristic, which carries no deadlock information at
all. The technique transfers in the narrow, real sense of "changes *when*
the search notices something's wrong," not in the sense of closing the
efficiency-mechanism gap between the two domains.

## What's not here

All of S1 (S1.1-1.5) is now covered. What's still deferred:

- **`giotto-tda`'s intrinsic-embedding persistence (S1.2 variant (b))** --
  shortest-path distance in the induced subgraph rather than `(g,h,f)`
  Euclidean distance. The doc scopes this as a follow-up after variant (a)
  exists on a handful of representative instances -- (a) is what's here.
- **Persistence landscapes/images for S1.2's population comparison** --
  used scalar summary statistics (bar count, total persistence) instead;
  `persim.PersLandscapeApprox`/`PersistenceImager` would let the population
  comparison work on the full diagram shape rather than two scalar
  summaries of it, at the cost of an unfamiliar library API.
- **Ollivier-Ricci curvature** -- needs an optimal-transport solver (`POT`,
  transitively installed by `GraphRicciCurvature` but not driven here); the
  doc's own prototype-scope note says start with Forman (no OT solver
  needed), which is what's here.
- **S3.3 (heuristic as natural transformation)** -- needs a post-hoc "true
  remaining cost" oracle per node, not just per-instance verification.
- **S3.4 (product/monoidal ablation)** -- scoped, not built: neither domain's
  heuristic actually has two components to ablate (`docs/DECISIONS.md` #16)
  -- Sokoban's cost model is push-count-only, so remaining cost is
  structurally independent of player position; HP's bound is entirely
  contact-count, no shape term exists either. Building components to force
  the test would mean inventing new heuristics from scratch, not ablating
  existing ones.
- **True per-node FRONTIER** -- would need a log call at every state
  generation regardless of outcome, which neither solver's hot loop has a free
  branch for.

Full scope and the original proposal:
`docs/equivalence/cross-domain-analysis-design.md`. Build rationale and the bugs
found while building this: `docs/DECISIONS.md` #11/#12/#14.